In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


def to_str(x):
    if isinstance(x, bytes):
        return x.decode("utf-8")
    return str(x)

def apply_log1p_selected(arr, feature_idx):
    arr = arr.copy()
    arr[:, :, feature_idx] = np.log1p(np.clip(arr[:, :, feature_idx], a_min=0, a_max=None))
    return arr

def invert_log1p_selected(arr, feature_idx):
    arr = arr.copy()
    arr[:, :, feature_idx] = np.expm1(arr[:, :, feature_idx])
    return arr

def mae_rmse(y_pred, y_true):
    mae = np.mean(np.abs(y_pred - y_true))
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    return mae, rmse

def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / (ss_tot + 1e-8)

class SolarForecastDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

class tAPE(nn.Module):
    def __init__(self, d_model, max_len=120, dropout=0.0, scale_factor=1.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin((position * div_term) * (d_model / max_len))
        pe[:, 1::2] = torch.cos((position * div_term) * (d_model / max_len))

        pe = scale_factor * pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerForecaster(nn.Module):
    def __init__(
        self,
        input_dim=8,
        seq_len=60,
        pred_len=60,
        d_model=128,
        nhead=4,
        num_layers=3,
        dim_feedforward=256,
        dropout=0.0,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.seq_len = seq_len
        self.pred_len = pred_len

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = tAPE(d_model=d_model, max_len=seq_len, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.Linear(seq_len * d_model, 512),
            nn.GELU(),
            nn.Linear(512, pred_len * input_dim),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.encoder(x)
        x = x.reshape(x.size(0), -1)
        out = self.head(x)
        out = out.view(x.size(0), self.pred_len, self.input_dim)
        return out

def raw_loss(pred, target, alpha=1.0, beta=0.1):
    horizon_weights = torch.linspace(1.0, 0.6, steps=pred.size(1), device=pred.device).view(1, pred.size(1), 1)
    point_loss = F.smooth_l1_loss(pred, target, reduction="none")
    point_loss = (point_loss * horizon_weights).mean()

    pred_diff = pred[:, 1:, :] - pred[:, :-1, :]
    target_diff = target[:, 1:, :] - target[:, :-1, :]
    diff_loss = torch.mean(torch.abs(pred_diff - target_diff))

    return alpha * point_loss + beta * diff_loss

def delta_loss(pred_delta, target_delta, alpha=1.0, beta=0.1):
    horizon_weights = torch.linspace(1.0, 0.6, steps=pred_delta.size(1), device=pred_delta.device).view(1, pred_delta.size(1), 1)
    point_loss = F.smooth_l1_loss(pred_delta, target_delta, reduction="none")
    point_loss = (point_loss * horizon_weights).mean()

    pred_diff = pred_delta[:, 1:, :] - pred_delta[:, :-1, :]
    target_diff = target_delta[:, 1:, :] - target_delta[:, :-1, :]
    diff_loss = torch.mean(torch.abs(pred_diff - target_diff))

    return alpha * point_loss + beta * diff_loss

def weighted_delta_loss(pred_delta, target_delta, alpha=1.0, beta=0.1, lam=2.0):
    horizon_weights = torch.linspace(1.0, 0.6, steps=pred_delta.size(1), device=pred_delta.device).view(1, pred_delta.size(1), 1)

    sample_weight = torch.mean(torch.abs(target_delta), dim=(1, 2))
    sample_weight = 1.0 + lam * sample_weight
    sample_weight = sample_weight / (sample_weight.mean() + 1e-8)
    sample_weight = sample_weight.view(-1, 1, 1)

    point_loss = F.smooth_l1_loss(pred_delta, target_delta, reduction="none")
    point_loss = point_loss * horizon_weights
    point_loss = point_loss * sample_weight
    point_loss = point_loss.mean()

    pred_diff = pred_delta[:, 1:, :] - pred_delta[:, :-1, :]
    target_diff = target_delta[:, 1:, :] - target_delta[:, :-1, :]
    diff_loss = torch.abs(pred_diff - target_diff)
    diff_loss = diff_loss * sample_weight[:, :1, :]
    diff_loss = diff_loss.mean()

    return alpha * point_loss + beta * diff_loss

def run_experiment(cfg):
    print("=" * 80)
    print("Running:", cfg["name"])
    print(cfg)
    print("=" * 80)

    np.random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)
    data = np.load(cfg["file_path"], allow_pickle=True)

    features = data["features"]          # (N, 60, 8)
    timestamps = data["timestamps"]      # (N, 60)
    ar_region = np.array([to_str(x) for x in data["ar_region"]])
    feature_names = [to_str(x) for x in data["feature_names"]]

    print("features shape:", features.shape)
    print("timestamps shape:", timestamps.shape)
    print("feature_names:", feature_names)

    start_time = pd.to_datetime([timestamps[i, 0] for i in range(len(timestamps))], errors="coerce")
    end_time   = pd.to_datetime([timestamps[i, -1] for i in range(len(timestamps))], errors="coerce")

    meta = pd.DataFrame({
        "row_idx": np.arange(len(features)),
        "ar_region": ar_region,
        "start_time": start_time,
        "end_time": end_time,
    }).dropna().copy()

    meta = meta.sort_values(["ar_region", "start_time"]).reset_index(drop=True)
    print("Rows after cleaning:", len(meta))
    if cfg["input_len"] == 60:
        meta["target_start"] = meta["start_time"] + pd.Timedelta(hours=cfg["forecast_gap_hours"])

        target_lookup = meta[["row_idx", "ar_region", "start_time"]].copy()
        target_lookup = target_lookup.rename(columns={
            "row_idx": "target_row_idx",
            "start_time": "target_start"
        })

        pairs = meta.merge(target_lookup, on=["ar_region", "target_start"], how="inner")
        pairs = pairs.sort_values(["ar_region", "start_time"]).reset_index(drop=True)

        print("Valid pairs:", len(pairs))

        unique_regions = np.array(sorted(pairs["ar_region"].unique()))
        rng = np.random.default_rng(cfg["seed"])
        rng.shuffle(unique_regions)

        n_regions = len(unique_regions)
        n_train = int(cfg["train_ratio"] * n_regions)
        n_val = int(cfg["val_ratio"] * n_regions)

        train_regions = unique_regions[:n_train]
        val_regions = unique_regions[n_train:n_train+n_val]
        test_regions = unique_regions[n_train+n_val:]

        train_df = pairs[pairs["ar_region"].isin(train_regions)].copy()
        val_df   = pairs[pairs["ar_region"].isin(val_regions)].copy()
        test_df  = pairs[pairs["ar_region"].isin(test_regions)].copy()

        X_train_raw = features[train_df["row_idx"].values]
        Y_train_raw = features[train_df["target_row_idx"].values]

        X_val_raw = features[val_df["row_idx"].values]
        Y_val_raw = features[val_df["target_row_idx"].values]

        X_test_raw = features[test_df["row_idx"].values]
        Y_test_raw = features[test_df["target_row_idx"].values]

    elif cfg["input_len"] == 120:
        meta["prev_start"] = meta["start_time"] - pd.Timedelta(hours=12)
        meta["target_start"] = meta["start_time"] + pd.Timedelta(hours=cfg["forecast_gap_hours"])

        prev_lookup = meta[["row_idx", "ar_region", "start_time", "end_time"]].copy()
        prev_lookup = prev_lookup.rename(columns={
            "row_idx": "prev_row_idx",
            "start_time": "prev_start",
            "end_time": "prev_end_time",
        })

        target_lookup = meta[["row_idx", "ar_region", "start_time"]].copy()
        target_lookup = target_lookup.rename(columns={
            "row_idx": "target_row_idx",
            "start_time": "target_start",
        })

        triples = meta.merge(prev_lookup, on=["ar_region", "prev_start"], how="inner")
        triples = triples.merge(target_lookup, on=["ar_region", "target_start"], how="inner")
        triples = triples[triples["prev_end_time"] + pd.Timedelta(minutes=12) == triples["start_time"]].copy()
        triples = triples.sort_values(["ar_region", "start_time"]).reset_index(drop=True)

        print("Valid triples:", len(triples))

        unique_regions = np.array(sorted(triples["ar_region"].unique()))
        rng = np.random.default_rng(cfg["seed"])
        rng.shuffle(unique_regions)

        n_regions = len(unique_regions)
        n_train = int(cfg["train_ratio"] * n_regions)
        n_val = int(cfg["val_ratio"] * n_regions)

        train_regions = unique_regions[:n_train]
        val_regions = unique_regions[n_train:n_train+n_val]
        test_regions = unique_regions[n_train+n_val:]

        train_df = triples[triples["ar_region"].isin(train_regions)].copy()
        val_df   = triples[triples["ar_region"].isin(val_regions)].copy()
        test_df  = triples[triples["ar_region"].isin(test_regions)].copy()

        X_train_raw = np.concatenate([
            features[train_df["prev_row_idx"].values],
            features[train_df["row_idx"].values]
        ], axis=1)
        Y_train_raw = features[train_df["target_row_idx"].values]

        X_val_raw = np.concatenate([
            features[val_df["prev_row_idx"].values],
            features[val_df["row_idx"].values]
        ], axis=1)
        Y_val_raw = features[val_df["target_row_idx"].values]

        X_test_raw = np.concatenate([
            features[test_df["prev_row_idx"].values],
            features[test_df["row_idx"].values]
        ], axis=1)
        Y_test_raw = features[test_df["target_row_idx"].values]
    else:
        raise ValueError("input_len must be 60 or 120")

    print("Before NaN filtering:")
    print("X_train:", X_train_raw.shape, "Y_train:", Y_train_raw.shape)
    print("X_val:", X_val_raw.shape, "Y_val:", Y_val_raw.shape)
    print("X_test:", X_test_raw.shape, "Y_test:", Y_test_raw.shape)

    # Drop NaN samples
    train_good = (~np.isnan(X_train_raw).any(axis=(1, 2))) & (~np.isnan(Y_train_raw).any(axis=(1, 2)))
    val_good   = (~np.isnan(X_val_raw).any(axis=(1, 2))) & (~np.isnan(Y_val_raw).any(axis=(1, 2)))
    test_good  = (~np.isnan(X_test_raw).any(axis=(1, 2))) & (~np.isnan(Y_test_raw).any(axis=(1, 2)))

    X_train_raw = X_train_raw[train_good]
    Y_train_raw = Y_train_raw[train_good]
    X_val_raw   = X_val_raw[val_good]
    Y_val_raw   = Y_val_raw[val_good]
    X_test_raw  = X_test_raw[test_good]
    Y_test_raw  = Y_test_raw[test_good]

    print("After NaN filtering:")
    print("X_train:", X_train_raw.shape, "Y_train:", Y_train_raw.shape)
    print("X_val:", X_val_raw.shape, "Y_val:", Y_val_raw.shape)
    print("X_test:", X_test_raw.shape, "Y_test:", Y_test_raw.shape)
    X_train_proc = X_train_raw.copy()
    Y_train_proc = Y_train_raw.copy()
    X_val_proc = X_val_raw.copy()
    Y_val_proc = Y_val_raw.copy()
    X_test_proc = X_test_raw.copy()
    Y_test_proc = Y_test_raw.copy()

    log_features = ["ABSNJZH", "SAVNCPP", "TOTBSQ", "TOTPOT", "TOTUSJH", "TOTUSJZ", "USFLUX"]
    log_feature_idx = [feature_names.index(f) for f in log_features]

    if cfg["use_log"]:
        X_train_proc = apply_log1p_selected(X_train_proc, log_feature_idx)
        Y_train_proc = apply_log1p_selected(Y_train_proc, log_feature_idx)
        X_val_proc   = apply_log1p_selected(X_val_proc, log_feature_idx)
        Y_val_proc   = apply_log1p_selected(Y_val_proc, log_feature_idx)
        X_test_proc  = apply_log1p_selected(X_test_proc, log_feature_idx)
        Y_test_proc  = apply_log1p_selected(Y_test_proc, log_feature_idx)


    train_mean = X_train_proc.reshape(-1, X_train_proc.shape[-1]).mean(axis=0)
    train_std  = X_train_proc.reshape(-1, X_train_proc.shape[-1]).std(axis=0) + 1e-8

    X_train_scaled = (X_train_proc - train_mean) / train_std
    X_val_scaled   = (X_val_proc - train_mean) / train_std
    X_test_scaled  = (X_test_proc - train_mean) / train_std

    Y_train_scaled = (Y_train_proc - train_mean) / train_std
    Y_val_scaled   = (Y_val_proc - train_mean) / train_std
    Y_test_scaled  = (Y_test_proc - train_mean) / train_std

    if cfg["predict_delta"]:
        Y_train_target = Y_train_scaled - X_train_scaled[:, -1:, :]
        Y_val_target   = Y_val_scaled - X_val_scaled[:, -1:, :]
        Y_test_target  = Y_test_scaled - X_test_scaled[:, -1:, :]
    else:
        Y_train_target = Y_train_scaled
        Y_val_target   = Y_val_scaled
        Y_test_target  = Y_test_scaled

    train_loader = DataLoader(SolarForecastDataset(X_train_scaled, Y_train_target), batch_size=cfg["batch_size"], shuffle=True)
    val_loader   = DataLoader(SolarForecastDataset(X_val_scaled, Y_val_target), batch_size=cfg["batch_size"], shuffle=False)
    test_loader  = DataLoader(SolarForecastDataset(X_test_scaled, Y_test_target), batch_size=cfg["batch_size"], shuffle=False)


    model = TransformerForecaster(
        input_dim=features.shape[-1],
        seq_len=cfg["input_len"],
        pred_len=60,
        d_model=cfg["d_model"],
        nhead=cfg["nhead"],
        num_layers=cfg["num_layers"],
        dim_feedforward=cfg["dim_ff"],
        dropout=cfg["dropout"],
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    def loss_fn(pred, target):
        if cfg["predict_delta"] and cfg["weighted_loss"]:
            return weighted_delta_loss(pred, target, alpha=1.0, beta=0.1, lam=2.0)
        elif cfg["predict_delta"]:
            return delta_loss(pred, target, alpha=1.0, beta=0.1)
        else:
            return raw_loss(pred, target, alpha=1.0, beta=0.1)

    def train_one_epoch():
        model.train()
        total_loss = 0.0
        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            optimizer.zero_grad()
            pred = model(X_batch)
            loss = loss_fn(pred, Y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * X_batch.size(0)
        return total_loss / len(train_loader.dataset)

    def eval_one_epoch(loader):
        model.eval()
        total_loss = 0.0
        with torch.no_grad():
            for X_batch, Y_batch in loader:
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)
                pred = model(X_batch)
                loss = loss_fn(pred, Y_batch)
                total_loss += loss.item() * X_batch.size(0)
        return total_loss / len(loader.dataset)

    best_val_loss = float("inf")
    patience_count = 0
    train_losses = []
    val_losses = []

    for epoch in range(cfg["max_epochs"]):
        train_loss = train_one_epoch()
        val_loss = eval_one_epoch(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Epoch {epoch+1}/{cfg['max_epochs']} | Train: {train_loss:.6f} | Val: {val_loss:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            torch.save(model.state_dict(), f"/content/{cfg['name']}.pt")
            print("Saved best model")
        else:
            patience_count += 1

        if patience_count >= cfg["patience"]:
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load(f"/content/{cfg['name']}.pt", map_location=device))
    model.eval()

    def get_preds(loader):
        preds = []
        trues = []
        with torch.no_grad():
            for X_batch, Y_batch in loader:
                X_batch = X_batch.to(device)
                pred = model(X_batch).cpu().numpy()
                preds.append(pred)
                trues.append(Y_batch.numpy())
        return np.concatenate(preds, axis=0), np.concatenate(trues, axis=0)

    pred_val_target, true_val_target = get_preds(val_loader)
    pred_test_target, true_test_target = get_preds(test_loader)

    if cfg["predict_delta"]:
        pred_val_scaled = pred_val_target + X_val_scaled[:, -1:, :]
        pred_test_scaled = pred_test_target + X_test_scaled[:, -1:, :]
    else:
        pred_val_scaled = pred_val_target
        pred_test_scaled = pred_test_target

    pred_val_proc = pred_val_scaled * train_std + train_mean
    pred_test_proc = pred_test_scaled * train_std + train_mean

    if cfg["use_log"]:
        pred_val_raw = invert_log1p_selected(pred_val_proc, log_feature_idx)
        pred_test_raw = invert_log1p_selected(pred_test_proc, log_feature_idx)
    else:
        pred_val_raw = pred_val_proc
        pred_test_raw = pred_test_proc

    # Raw truth and persistence baseline
    true_val_raw = Y_val_raw
    true_test_raw = Y_test_raw

    baseline_val_raw = np.repeat(X_val_raw[:, -1:, :], repeats=60, axis=1)
    baseline_test_raw = np.repeat(X_test_raw[:, -1:, :], repeats=60, axis=1)

    # Metrics
    val_model_mae, val_model_rmse = mae_rmse(pred_val_raw, true_val_raw)
    val_base_mae, val_base_rmse = mae_rmse(baseline_val_raw, true_val_raw)

    test_model_mae, test_model_rmse = mae_rmse(pred_test_raw, true_test_raw)
    test_base_mae, test_base_rmse = mae_rmse(baseline_test_raw, true_test_raw)

    if cfg["predict_delta"]:
        delta_val_mae, delta_val_rmse = mae_rmse(pred_val_target, true_val_target)
        delta_test_mae, delta_test_rmse = mae_rmse(pred_test_target, true_test_target)
    else:
        delta_val_mae, delta_val_rmse = None, None
        delta_test_mae, delta_test_rmse = None, None

    per_feature_mae_model = np.mean(np.abs(pred_test_raw - true_test_raw), axis=(0, 1))
    per_feature_mae_base = np.mean(np.abs(baseline_test_raw - true_test_raw), axis=(0, 1))
    per_feature_r2 = []
    for i in range(len(feature_names)):
        y_true_f = true_test_raw[:, :, i].reshape(-1)
        y_pred_f = pred_test_raw[:, :, i].reshape(-1)
        per_feature_r2.append(r2_score_manual(y_true_f, y_pred_f))
    per_feature_r2 = np.array(per_feature_r2)

    per_horizon_model = np.mean(np.abs(pred_test_raw - true_test_raw), axis=(0, 2))
    per_horizon_base = np.mean(np.abs(baseline_test_raw - true_test_raw), axis=(0, 2))

    print("\nOVERALL")
    if cfg["predict_delta"]:
        print("Delta-space Val MAE/RMSE :", delta_val_mae, delta_val_rmse)
        print("Delta-space Test MAE/RMSE:", delta_test_mae, delta_test_rmse)

    print("\nValidation")
    print("Model       MAE:", val_model_mae, "RMSE:", val_model_rmse)
    print("Persistence MAE:", val_base_mae, "RMSE:", val_base_rmse)

    print("\nTest")
    print("Model       MAE:", test_model_mae, "RMSE:", test_model_rmse)
    print("Persistence MAE:", test_base_mae, "RMSE:", test_base_rmse)

    print("\nPER-FEATURE R2")
    for name, r2 in zip(feature_names, per_feature_r2):
        print(f"{name:10s} | R2: {r2:.6f}")

    print("\nPER-FEATURE MAE")
    for name, m, b in zip(feature_names, per_feature_mae_model, per_feature_mae_base):
        print(f"{name:10s} | Model: {m:.6f} | Persistence: {b:.6f} | Gain: {b - m:.6f}")

    print("\nFIRST 20 HORIZONS")
    for i in range(min(20, len(per_horizon_model))):
        print(f"Step {i+1:02d} | Model: {per_horizon_model[i]:.6f} | Persistence: {per_horizon_base[i]:.6f} | Gain: {per_horizon_base[i] - per_horizon_model[i]:.6f}")

    results = {
        "name": cfg["name"],
        "val_model_mae": float(val_model_mae),
        "val_model_rmse": float(val_model_rmse),
        "val_persistence_mae": float(val_base_mae),
        "val_persistence_rmse": float(val_base_rmse),
        "test_model_mae": float(test_model_mae),
        "test_model_rmse": float(test_model_rmse),
        "test_persistence_mae": float(test_base_mae),
        "test_persistence_rmse": float(test_base_rmse),
        "per_feature_r2": {k: float(v) for k, v in zip(feature_names, per_feature_r2)},
    }
    return results

In [ ]:
exp1 = run_experiment({
    "name": "exp1_12h_60step_raw",
    "file_path": "/content/partition1_grouped.npz",
    "seed": 42,
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "forecast_gap_hours": 12,
    "input_len": 60,
    "use_log": False,
    "predict_delta": False,
    "weighted_loss": False,
    "batch_size": 64,
    "max_epochs": 30,
    "patience": 6,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "d_model": 128,
    "nhead": 4,
    "num_layers": 3,
    "dim_ff": 256,
    "dropout": 0.0,
})

Running: exp1_12h_60step_raw
{'name': 'exp1_12h_60step_raw', 'file_path': '/content/partition1_grouped.npz', 'seed': 42, 'train_ratio': 0.7, 'val_ratio': 0.15, 'forecast_gap_hours': 12, 'input_len': 60, 'use_log': False, 'predict_delta': False, 'weighted_loss': False, 'batch_size': 64, 'max_epochs': 30, 'patience': 6, 'lr': 0.001, 'weight_decay': 1e-05, 'd_model': 128, 'nhead': 4, 'num_layers': 3, 'dim_ff': 256, 'dropout': 0.0}
Device: cuda
features shape: (73492, 60, 8)
timestamps shape: (73492, 60)
feature_names: ['ABSNJZH', 'SAVNCPP', 'TOTBSQ', 'TOTPOT', 'TOTUSJH', 'TOTUSJZ', 'USFLUX', 'R_VALUE']
Rows after cleaning: 73492
Valid pairs: 57417
Before NaN filtering:
X_train: (40138, 60, 8) Y_train: (40138, 60, 8)
X_val: (8836, 60, 8) Y_val: (8836, 60, 8)
X_test: (8443, 60, 8) Y_test: (8443, 60, 8)
After NaN filtering:
X_train: (36838, 60, 8) Y_train: (36838, 60, 8)
X_val: (8266, 60, 8) Y_val: (8266, 60, 8)
X_test: (7735, 60, 8) Y_test: (7735, 60, 8)
Epoch 1/30 | Train: 0.040397 | Val: 

In [ ]:
exp2 = run_experiment({
    "name": "exp2_24h_60step_delta_log",
    "file_path": "/content/partition1_grouped.npz",
    "seed": 42,
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "forecast_gap_hours": 24,
    "input_len": 60,
    "use_log": True,
    "predict_delta": True,
    "weighted_loss": False,
    "batch_size": 64,
    "max_epochs": 50,
    "patience": 8,
    "lr": 3e-4,
    "weight_decay": 1e-5,
    "d_model": 128,
    "nhead": 4,
    "num_layers": 3,
    "dim_ff": 256,
    "dropout": 0.0,
})

Running: exp2_24h_60step_delta_log
{'name': 'exp2_24h_60step_delta_log', 'file_path': '/content/partition1_grouped.npz', 'seed': 42, 'train_ratio': 0.7, 'val_ratio': 0.15, 'forecast_gap_hours': 24, 'input_len': 60, 'use_log': True, 'predict_delta': True, 'weighted_loss': False, 'batch_size': 64, 'max_epochs': 50, 'patience': 8, 'lr': 0.0003, 'weight_decay': 1e-05, 'd_model': 128, 'nhead': 4, 'num_layers': 3, 'dim_ff': 256, 'dropout': 0.0}
Device: cuda
features shape: (73492, 60, 8)
timestamps shape: (73492, 60)
feature_names: ['ABSNJZH', 'SAVNCPP', 'TOTBSQ', 'TOTPOT', 'TOTUSJH', 'TOTUSJZ', 'USFLUX', 'R_VALUE']
Rows after cleaning: 73492
Valid pairs: 53378
Before NaN filtering:
X_train: (37956, 60, 8) Y_train: (37956, 60, 8)
X_val: (7662, 60, 8) Y_val: (7662, 60, 8)
X_test: (7760, 60, 8) Y_test: (7760, 60, 8)
After NaN filtering:
X_train: (34965, 60, 8) Y_train: (34965, 60, 8)
X_val: (7073, 60, 8) Y_val: (7073, 60, 8)
X_test: (7179, 60, 8) Y_test: (7179, 60, 8)
Epoch 1/50 | Train: 0.070

In [ ]:
exp3 = run_experiment({
    "name": "exp3_24h_60step_delta_log_weighted",
    "file_path": "/content/partition1_grouped.npz",
    "seed": 42,
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "forecast_gap_hours": 24,
    "input_len": 60,
    "use_log": True,
    "predict_delta": True,
    "weighted_loss": True,
    "batch_size": 64,
    "max_epochs": 50,
    "patience": 8,
    "lr": 3e-4,
    "weight_decay": 1e-5,
    "d_model": 128,
    "nhead": 4,
    "num_layers": 3,
    "dim_ff": 256,
    "dropout": 0.0,
})

Running: exp3_24h_60step_delta_log_weighted
{'name': 'exp3_24h_60step_delta_log_weighted', 'file_path': '/content/partition1_grouped.npz', 'seed': 42, 'train_ratio': 0.7, 'val_ratio': 0.15, 'forecast_gap_hours': 24, 'input_len': 60, 'use_log': True, 'predict_delta': True, 'weighted_loss': True, 'batch_size': 64, 'max_epochs': 50, 'patience': 8, 'lr': 0.0003, 'weight_decay': 1e-05, 'd_model': 128, 'nhead': 4, 'num_layers': 3, 'dim_ff': 256, 'dropout': 0.0}
Device: cuda
features shape: (73492, 60, 8)
timestamps shape: (73492, 60)
feature_names: ['ABSNJZH', 'SAVNCPP', 'TOTBSQ', 'TOTPOT', 'TOTUSJH', 'TOTUSJZ', 'USFLUX', 'R_VALUE']
Rows after cleaning: 73492
Valid pairs: 53378
Before NaN filtering:
X_train: (37956, 60, 8) Y_train: (37956, 60, 8)
X_val: (7662, 60, 8) Y_val: (7662, 60, 8)
X_test: (7760, 60, 8) Y_test: (7760, 60, 8)
After NaN filtering:
X_train: (34965, 60, 8) Y_train: (34965, 60, 8)
X_val: (7073, 60, 8) Y_val: (7073, 60, 8)
X_test: (7179, 60, 8) Y_test: (7179, 60, 8)
Epoch 1/

In [ ]:
exp4 = run_experiment({
    "name": "exp4_24h_120step_delta_log",
    "file_path": "/content/partition1_grouped.npz",
    "seed": 42,
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "forecast_gap_hours": 24,
    "input_len": 120,
    "use_log": True,
    "predict_delta": True,
    "weighted_loss": False,
    "batch_size": 64,
    "max_epochs": 50,
    "patience": 8,
    "lr": 3e-4,
    "weight_decay": 1e-5,
    "d_model": 128,
    "nhead": 4,
    "num_layers": 3,
    "dim_ff": 256,
    "dropout": 0.0,
})

Running: exp4_24h_120step_delta_log
{'name': 'exp4_24h_120step_delta_log', 'file_path': '/content/partition1_grouped.npz', 'seed': 42, 'train_ratio': 0.7, 'val_ratio': 0.15, 'forecast_gap_hours': 24, 'input_len': 120, 'use_log': True, 'predict_delta': True, 'weighted_loss': False, 'batch_size': 64, 'max_epochs': 50, 'patience': 8, 'lr': 0.0003, 'weight_decay': 1e-05, 'd_model': 128, 'nhead': 4, 'num_layers': 3, 'dim_ff': 256, 'dropout': 0.0}
Device: cuda
features shape: (73492, 60, 8)
timestamps shape: (73492, 60)
feature_names: ['ABSNJZH', 'SAVNCPP', 'TOTBSQ', 'TOTPOT', 'TOTUSJH', 'TOTUSJZ', 'USFLUX', 'R_VALUE']
Rows after cleaning: 73492
Valid triples: 42079
Before NaN filtering:
X_train: (29434, 120, 8) Y_train: (29434, 60, 8)
X_val: (5871, 120, 8) Y_val: (5871, 60, 8)
X_test: (6774, 120, 8) Y_test: (6774, 60, 8)
After NaN filtering:
X_train: (26609, 120, 8) Y_train: (26609, 60, 8)
X_val: (5242, 120, 8) Y_val: (5242, 60, 8)
X_test: (5950, 120, 8) Y_test: (5950, 60, 8)
Epoch 1/50 | T

In [ ]:
summary_df = pd.DataFrame([
    {
        "Experiment": exp1["name"],
        "Test MAE": exp1["test_model_mae"],
        "Test RMSE": exp1["test_model_rmse"],
        "Persistence Test MAE": exp1["test_persistence_mae"],
        "Persistence Test RMSE": exp1["test_persistence_rmse"],
        **{f"R2_{k}": v for k, v in exp1["per_feature_r2"].items()},
    },
    {
        "Experiment": exp2["name"],
        "Test MAE": exp2["test_model_mae"],
        "Test RMSE": exp2["test_model_rmse"],
        "Persistence Test MAE": exp2["test_persistence_mae"],
        "Persistence Test RMSE": exp2["test_persistence_rmse"],
        **{f"R2_{k}": v for k, v in exp2["per_feature_r2"].items()},
    },
    {
        "Experiment": exp3["name"],
        "Test MAE": exp3["test_model_mae"],
        "Test RMSE": exp3["test_model_rmse"],
        "Persistence Test MAE": exp3["test_persistence_mae"],
        "Persistence Test RMSE": exp3["test_persistence_rmse"],
        **{f"R2_{k}": v for k, v in exp3["per_feature_r2"].items()},
    },
    {
        "Experiment": exp4["name"],
        "Test MAE": exp4["test_model_mae"],
        "Test RMSE": exp4["test_model_rmse"],
        "Persistence Test MAE": exp4["test_persistence_mae"],
        "Persistence Test RMSE": exp4["test_persistence_rmse"],
        **{f"R2_{k}": v for k, v in exp4["per_feature_r2"].items()},
    },
])

print(summary_df)

                           Experiment      Test MAE     Test RMSE  \
0                 exp1_12h_60step_raw  1.765826e+21  1.383977e+22   
1           exp2_24h_60step_delta_log  2.235133e+21  1.995695e+22   
2  exp3_24h_60step_delta_log_weighted  2.232697e+21  1.585814e+22   
3          exp4_24h_120step_delta_log  1.056158e+21  6.299273e+21   

   Persistence Test MAE  Persistence Test RMSE  R2_ABSNJZH  R2_SAVNCPP  \
0          9.726700e+20           6.519862e+21    0.867726    0.865369   
1          2.368128e+21           1.614846e+22    0.529686    0.493469   
2          2.368128e+21           1.614846e+22    0.672445    0.665350   
3          1.180178e+21           6.359821e+21    0.714198    0.686455   

   R2_TOTBSQ  R2_TOTPOT  R2_TOTUSJH  R2_TOTUSJZ  R2_USFLUX  R2_R_VALUE  
0   0.984254   0.978667    0.972927    0.975758   0.981903    0.880594  
1   0.977524   0.955003    0.963620    0.965805   0.974637    0.796011  
2   0.975757   0.971619    0.962163    0.964569   0.970458    0.

In [ ]:
import pandas as pd

def build_row(label, horizon, input_len, exp):
    model_mae = exp["test_model_mae"]
    model_rmse = exp["test_model_rmse"]
    base_mae = exp["test_persistence_mae"]
    base_rmse = exp["test_persistence_rmse"]

    mae_gain = base_mae - model_mae
    rmse_gain = base_rmse - model_rmse

    mae_pct_gain = 100 * mae_gain / base_mae
    rmse_pct_gain = 100 * rmse_gain / base_rmse

    return {
        "Experiment": label,
        "Horizon": horizon,
        "Input": input_len,
        "Test MAE": model_mae,
        "Persistence MAE": base_mae,
        "MAE Gain": mae_gain,
        "MAE % Gain": mae_pct_gain,
        "Test RMSE": model_rmse,
        "Persistence RMSE": base_rmse,
        "RMSE Gain": rmse_gain,
        "RMSE % Gain": rmse_pct_gain,
    }

poster_gain_table = pd.DataFrame([
    build_row("E1", "12h", 60, exp1),
    build_row("E2", "24h", 60, exp2),
    build_row("E3", "24h", 60, exp3),
    build_row("E4", "24h", 120, exp4),
])

# print(poster_gain_table)

poster_gain_table_display = poster_gain_table.copy()

for col in ["Test MAE", "Persistence MAE", "MAE Gain", "Test RMSE", "Persistence RMSE", "RMSE Gain"]:
    poster_gain_table_display[col] = poster_gain_table_display[col].map(lambda x: f"{x:.3e}")

for col in ["MAE % Gain", "RMSE % Gain"]:
    poster_gain_table_display[col] = poster_gain_table_display[col].map(lambda x: f"{x:.1f}%")

print(poster_gain_table_display)

  Experiment Horizon  Input   Test MAE Persistence MAE    MAE Gain MAE % Gain  \
0         E1     12h     60  1.766e+21       9.727e+20  -7.932e+20     -81.5%   
1         E2     24h     60  2.235e+21       2.368e+21   1.330e+20       5.6%   
2         E3     24h     60  2.233e+21       2.368e+21   1.354e+20       5.7%   
3         E4     24h    120  1.056e+21       1.180e+21   1.240e+20      10.5%   

   Test RMSE Persistence RMSE   RMSE Gain RMSE % Gain  
0  1.384e+22        6.520e+21  -7.320e+21     -112.3%  
1  1.996e+22        1.615e+22  -3.808e+21      -23.6%  
2  1.586e+22        1.615e+22   2.903e+20        1.8%  
3  6.299e+21        6.360e+21   6.055e+19        1.0%  
